# Analyse des Données de Ventes (Retail Data)
Ce notebook explore et nettoie un jeu de données de transactions.

In [ ]:
import pandas as pd
import numpy as np
# Import unique de display et Markdown
from IPython.display import display, Markdown

# Chargement des données (à adapter avec ton chemin de fichier)
df = pd.read_csv('ton_fichier.csv')

## 1.1 Dataset Summary
Aperçu général des données (taille, premières lignes).

## 1.2 Vérification de la cohérence des types (Column types)
Avant d'effectuer des calculs, nous devons nous assurer que les identifiants de factures (`Invoice`) sont homogènes. Une exploration proactive permet de vérifier si cette colonne contient des caractères non-numériques (comme des lettres), ce qui indiquerait une nomenclature spécifique.

In [ ]:
# Détection proactive des factures contenant autre chose que des chiffres
non_numeric_mask = ~df['Invoice'].astype(str).str.isnumeric()
non_numeric_invoices = df[non_numeric_mask]

display(Markdown(f"**Nombre de factures avec des caractères non-numériques :** {len(non_numeric_invoices)}"))
display(non_numeric_invoices.head())

# On observe que ces factures commencent par "C".

**Observation :** Les factures non-numériques commencent par la lettre "C". En regardant les quantités associées, on remarque qu'elles sont négatives. Cela indique qu'il s'agit de transactions d'annulation (C = Cancellation).

In [ ]:
## 1.3 Data Cleaning & Fiabilité des données
Nettoyage des anomalies, valeurs manquantes et incohérences.

In [ ]:
# 1. Traitement de l'anomalie d'annulation (C) avec quantité positive
# On exclut explicitement toute ligne commençant par C qui aurait une quantité > 0
anomalie_c_positive = (df['Invoice'].astype(str).str.startswith('C')) & (df['Quantity'] > 0)
df = df[~anomalie_c_positive]

# 2. Vérification des quantités négatives qui ne sont PAS des annulations
qty_neg_sans_c = df[(df['Quantity'] < 0) & (~df['Invoice'].astype(str).str.startswith('C'))]
if not qty_neg_sans_c.empty:
    print(f"Attention : {len(qty_neg_sans_c)} lignes ont des quantités négatives sans préfixe 'C'.")
    # Action à définir (ex: les supprimer ou les corriger)

# 3. Vérification des prix égaux à 0 (articles gratuits ou erreurs)
prix_zero = df[df['Price'] == 0]
if not prix_zero.empty:
    print(f"Note : {len(prix_zero)} lignes ont un prix égal à 0.")
    # On peut décider de les garder si ce sont des cadeaux, ou de les exclure.
    
# 4. Suppression des valeurs manquantes (Customer ID)
df = df.dropna(subset=['Customer ID'])

In [ ]:
# Exemple de correction pour l'analyse des codes non-produits
# Ajout de include_groups=False pour anticiper la nouvelle version de Pandas
def analyser_codes(group):
    # Logique interne de ta fonction
    return group['Quantity'].sum()

# Le paramètre include_groups=False évite le FutureWarning
resultats = df.groupby('Invoice').apply(analyser_codes, include_groups=False)